# 05 视频分析与目标跟踪

## 概述

视频分析是计算机视觉中的重要领域，涉及从视频序列中检测、跟踪和分析运动物体。本模块使用合成视频（避免外部文件依赖）来演示核心算法。

学习目标：
1. 使用 numpy 生成合成视频帧
2. 掌握 Lucas-Kanade 稀疏光流
3. 理解 Farneback 密集光流
4. 使用背景减除法检测运动物体
5. 实现基本的目标跟踪（centroid tracking）
6. 模拟交通流量监测场景

> 注：本模块所有"视频"均由 numpy 合成，无需下载外部视频文件。

In [ ]:
# ===== 环境准备 =====
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Circle, Rectangle as RectPatch
from ipywidgets import interact, IntSlider, FloatSlider
import cv2

from utils.image_utils import (load_image, display_image, display_images,
                                compare_images, ensure_uint8, ensure_float)
from utils.sample_data import checkerboard
from utils.visualization import plot_histogram

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
print('环境准备完毕！')
print(f'OpenCV 版本: {cv2.__version__}')

## 生成合成视频

在实际项目中，我们通常处理从摄像头或文件中读取的视频。为了在任何环境下都能运行本教程，我们使用 numpy 生成合成视频帧来模拟真实场景。

合成视频的优势：
- 不依赖外部文件
- 可以精确控制物体的运动参数
- 便于验证算法的准确性（ground truth 已知）

下面我们生成一个简单的合成视频：一个白色矩形在黑色背景上匀速移动。

In [ ]:
# ===== 生成合成视频帧 =====

def generate_moving_rect_frames(num_frames=50, frame_size=(200, 200),
                                  rect_size=(30, 30), velocity=(3, 2)):
    """生成白色矩形在黑色背景上移动的视频帧"""
    frames = []
    positions = []

    x, y = 20, 20  # 初始位置

    for i in range(num_frames):
        frame = np.zeros((frame_size[0], frame_size[1]), dtype=np.uint8)
        # 确保矩形在画面内
        x = x + velocity[0]
        y = y + velocity[1]

        # 边界反弹
        if x < 0 or x + rect_size[0] >= frame_size[1]:
            velocity = (-velocity[0], velocity[1])
            x = max(0, min(x, frame_size[1] - rect_size[0] - 1))
        if y < 0 or y + rect_size[0] >= frame_size[0]:
            velocity = (velocity[0], -velocity[1])
            y = max(0, min(y, frame_size[0] - rect_size[0] - 1))

        cv2.rectangle(frame, (x, y), (x + rect_size[0], y + rect_size[0]), 255, -1)
        frames.append(frame)
        positions.append((x, y))

    return frames, positions


# 生成 50 帧视频
frames_rect, pos_rect = generate_moving_rect_frames(num_frames=50,
                                                       frame_size=(200, 200),
                                                       rect_size=(30, 30),
                                                       velocity=(3, 2))

print(f'生成帧数: {len(frames_rect)}')
print(f'帧尺寸: {frames_rect[0].shape}')
print(f'初始位置: {pos_rect[0]}')
print(f'最终位置: {pos_rect[-1]}')

# 显示前 4 帧
sample_indices = [0, 15, 30, 45]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, idx in zip(axes, sample_indices):
    ax.imshow(frames_rect[idx], cmap='gray')
    ax.set_title(f'帧 {idx} (位置: {pos_rect[idx]})')
    ax.axis('off')
plt.suptitle('合成视频帧（匀速运动矩形）', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ===== 生成弹跳球的合成视频 =====

def generate_bouncing_ball_frames(num_frames=60, frame_size=(300, 300),
                                    ball_radius=20, velocity=(5, 3)):
    """生成弹跳球的合成视频帧（彩色）"""
    frames = []
    positions = []

    x = frame_size[1] // 4
    y = frame_size[0] // 2
    vx, vy = velocity

    for i in range(num_frames):
        frame = np.zeros((frame_size[0], frame_size[1], 3), dtype=np.uint8)

        x += vx
        y += vy

        # 边界反弹
        if x - ball_radius < 0:
            x = ball_radius
            vx = abs(vx)
        if x + ball_radius >= frame_size[1]:
            x = frame_size[1] - ball_radius - 1
            vx = -abs(vx)
        if y - ball_radius < 0:
            y = ball_radius
            vy = abs(vy)
        if y + ball_radius >= frame_size[0]:
            y = frame_size[0] - ball_radius - 1
            vy = -abs(vy)

        # 画红色弹跳球
        cv2.circle(frame, (int(x), int(y)), ball_radius, (0, 0, 255), -1)
        # 添加高光
        cv2.circle(frame, (int(x - ball_radius//3), int(y - ball_radius//3)),
                   ball_radius // 4, (255, 200, 200), -1)

        frames.append(frame)
        positions.append((x, y))

    return frames, positions


# 生成弹跳球视频
ball_frames, ball_positions = generate_bouncing_ball_frames(
    num_frames=60, frame_size=(300, 300), ball_radius=20, velocity=(5, 3)
)

print(f'弹跳球帧数: {len(ball_frames)}')

# 显示 4 帧
sample_idx = [0, 18, 35, 50]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, idx in zip(axes, sample_idx):
    ax.imshow(ball_frames[idx])
    x, y = ball_positions[idx]
    ax.set_title(f'帧 {idx}\n位置: ({x:.0f}, {y:.0f})')
    ax.axis('off')
plt.suptitle('弹跳球合成视频——4帧展示', fontsize=14)
plt.tight_layout()
plt.show()

# 绘制弹跳球的运动轨迹
positions_arr = np.array(ball_positions)
plt.figure(figsize=(8, 6))
plt.plot(positions_arr[:, 0], positions_arr[:, 1], 'b-', alpha=0.6, linewidth=1)
plt.scatter(positions_arr[0, 0], positions_arr[0, 1], c='green', s=100, label='起点', zorder=5)
plt.scatter(positions_arr[-1, 0], positions_arr[-1, 1], c='red', s=100, label='终点', zorder=5)
plt.xlim(0, 300)
plt.ylim(300, 0)  # 翻转 y 轴匹配图像坐标
plt.xlabel('X')
plt.ylabel('Y')
plt.title('弹跳球运动轨迹')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Lucas-Kanade 稀疏光流

光流（Optical Flow）是空间运动物体在观测成像面上的像素运动的瞬时速度。Lucas-Kanade 方法是经典的稀疏光流算法。

### 核心假设
1. **亮度恒定**：同一空间点的像素值在不同帧中保持不变
2. **小运动**：相邻帧之间物体运动较小
3. **空间一致性**：邻域内像素具有相同的运动

### 算法流程
1. `cv2.goodFeaturesToTrack` — 在第一帧中检测好的角点（Shi-Tomasi）
2. `cv2.calcOpticalFlowPyrLK` — 使用金字塔 Lucas-Kanade 方法追踪这些角点
3. 绘制每个角点从第一帧到当前帧的运动轨迹

In [ ]:
# ===== Lucas-Kanade 稀疏光流 =====

# 准备两帧（间隔 3 帧以看到明显运动）
frame0 = frames_rect[0]
frame1 = frames_rect[3]

# 在第一帧中检测角点
feature_params = dict(maxCorners=30, qualityLevel=0.3, minDistance=10, blockSize=7)
corners0 = cv2.goodFeaturesToTrack(frame0, mask=None, **feature_params)

if corners0 is not None:
    print(f'在帧 0 中检测到 {len(corners0)} 个角点')
else:
    print('未检测到角点，调整参数...')
    corners0 = cv2.goodFeaturesToTrack(frame0, mask=None, maxCorners=20,
                                         qualityLevel=0.1, minDistance=5, blockSize=7)
    print(f'重新检测到 {len(corners0) if corners0 is not None else 0} 个角点')

# LK 光流追踪
lk_params = dict(winSize=(15, 15), maxLevel=2,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))
corners1, status, _ = cv2.calcOpticalFlowPyrLK(frame0, frame1, corners0, None, **lk_params)

# 筛选成功追踪的点
good_corners0 = corners0[status == 1]
good_corners1 = corners1[status == 1]
print(f'成功追踪的点数: {len(good_corners0)} / {len(corners0)}')

# 可视化追踪结果
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 帧 0 — 检测到的角点
axes[0].imshow(frame0, cmap='gray')
for pt in corners0:
    axes[0].plot(pt[0][0], pt[0][1], 'go', markersize=4)
axes[0].set_title(f'帧 0 — 检测到的角点 ({len(corners0)}个)')
axes[0].axis('off')

# 帧 1 — 追踪结果 + 运动向量
axes[1].imshow(frame1, cmap='gray')
for pt0, pt1 in zip(good_corners0, good_corners1):
    x0, y0 = pt0[0]
    x1, y1 = pt1[0]
    axes[1].arrow(x0, y0, x1 - x0, y1 - y0,
                  head_width=3, head_length=4, fc='red', ec='red', alpha=0.7)
    axes[1].plot(x1, y1, 'ro', markersize=3)
axes[1].set_title(f'帧 3 — LK 光流追踪 ({len(good_corners0)}条轨迹)')
axes[1].axis('off')

plt.suptitle('Lucas-Kanade 稀疏光流', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ===== 弹跳球视频上的 LK 光流 =====

# 使用弹跳球的帧
bf0 = cv2.cvtColor(ball_frames[0], cv2.COLOR_RGB2GRAY)
bf1 = cv2.cvtColor(ball_frames[3], cv2.COLOR_RGB2GRAY)

# 检测角点
corners_ball = cv2.goodFeaturesToTrack(bf0, mask=None, maxCorners=50,
                                         qualityLevel=0.01, minDistance=8, blockSize=7)

if corners_ball is not None:
    corners_ball_1, status_b, _ = cv2.calcOpticalFlowPyrLK(
        bf0, bf1, corners_ball, None, **lk_params
    )
    good_old = corners_ball[status_b == 1]
    good_new = corners_ball_1[status_b == 1]
    print(f'弹跳球：检测到 {len(corners_ball)} 个角点，追踪到 {len(good_old)} 个')
else:
    good_old, good_new = np.array([]), np.array([])
    print('弹跳球：未检测到足够角点，请在球体上增加纹理')

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(ball_frames[0])
for pt in (corners_ball or []):
    axes[0].plot(pt[0][0], pt[0][1], 'g+', markersize=6)
axes[0].set_title(f'帧 0 — 角点检测')
axes[0].axis('off')

axes[1].imshow(ball_frames[3])
for pt0, pt1 in zip(good_old, good_new):
    axes[1].arrow(pt0[0][0], pt0[0][1], pt1[0][0] - pt0[0][0], pt1[0][1] - pt0[0][1],
                  head_width=3, head_length=4, fc='yellow', ec='yellow', alpha=0.8)
axes[1].set_title(f'帧 3 — LK 光流追踪')
axes[1].axis('off')

plt.suptitle('弹跳球上的 Lucas-Kanade 光流', fontsize=14)
plt.tight_layout()
plt.show()

## Farneback 密集光流

与稀疏光流不同，密集光流计算图像中**每个像素**的运动向量。Gunnar Farneback 的算法基于多项式展开来近似每个像素的邻域，然后通过分析多项式的平移来估计位移。

### 可视化方法
使用 HSV 颜色空间将光流场转换为彩色图像：
- **色调 (Hue)**：表示运动方向（0-360度对应不同方向）
- **饱和度 (Saturation)** / **亮度 (Value)**：表示运动幅度

这样，静止区域为白色，不同方向的运动呈现不同颜色，色彩越鲜艳表示运动越剧烈。

In [ ]:
# ===== Farneback 密集光流 =====

# 使用弹跳球的连续两帧
frame_a = cv2.cvtColor(ball_frames[2], cv2.COLOR_RGB2GRAY)
frame_b = cv2.cvtColor(ball_frames[3], cv2.COLOR_RGB2GRAY)

# 计算密集光流
flow = cv2.calcOpticalFlowFarneback(
    frame_a, frame_b, None,
    pyr_scale=0.5,   # 金字塔缩放
    levels=3,        # 金字塔层数
    winsize=15,      # 窗口大小
    iterations=3,    # 迭代次数
    poly_n=5,        # 多项式展开的邻域大小
    poly_sigma=1.2,  # 高斯标准差
    flags=0
)

# 将光流转换为 HSV 颜色编码
mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
hsv = np.zeros((frame_a.shape[0], frame_a.shape[1], 3), dtype=np.uint8)
hsv[..., 0] = ang * 180 / np.pi / 2   # 色调 = 方向
hsv[..., 1] = 255                       # 饱和度 = 最大
hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)  # 亮度 = 幅度
flow_rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

# 可视化
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].imshow(ball_frames[2])
axes[0, 0].set_title('帧 2（前一帧）')
axes[0, 0].axis('off')

axes[0, 1].imshow(ball_frames[3])
axes[0, 1].set_title('帧 3（后一帧）')
axes[0, 1].axis('off')

axes[1, 0].imshow(flow_rgb)
axes[1, 0].set_title('密集光流（HSV 编码）\n颜色=方向, 亮度=速度')
axes[1, 0].axis('off')

axes[1, 1].imshow(mag, cmap='hot')
axes[1, 1].set_title('运动幅度图（光流大小）')
axes[1, 1].axis('off')

plt.suptitle('Farneback 密集光流', fontsize=14)
plt.tight_layout()
plt.show()

print('光流统计:')
print(f'  最大位移: {mag.max():.2f} 像素')
print(f'  平均位移: {mag.mean():.2f} 像素')
print(f'  非零像素占比: {(mag > 0.5).mean() * 100:.1f}%')

## 背景减除法

背景减除（Background Subtraction）是运动检测的经典方法。核心思想是建立一个背景模型，然后将每一帧与背景模型比较，差异显著的区域即为前景（运动物体）。

OpenCV 提供了几种背景减除算法：
- **MOG2** (Mixture of Gaussians 2)：对每个像素维护多个高斯分布，自适应适应光照变化和场景变动
- **KNN**：使用 K 近邻方法建模背景
- **GMG**：结合贝叶斯推断

MOG2 的优点是能自动选择高斯分量的数量，并具有阴影检测能力。

In [ ]:
# ===== 交互式光流参数探索 =====

from ipywidgets import interact, IntSlider, FloatSlider, Dropdown

fr0 = frames_rect[0]
fr1 = frames_rect[3]

def explore_lk_flow(max_corners=50, quality=0.3, min_dist=7, win_size=15, max_level=2):
    """交互式探索 LK 光流参数"""
    fp = dict(maxCorners=max_corners, qualityLevel=quality,
              minDistance=min_dist, blockSize=7)
    p0 = cv2.goodFeaturesToTrack(fr0, mask=None, **fp)

    if p0 is None:
        print('未检测到足够特征点，请降低 quality level')
        return

    lkp = dict(winSize=(win_size, win_size), maxLevel=max_level,
               criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))
    p1, st, err = cv2.calcOpticalFlowPyrLK(fr0, fr1, p0, None, **lkp)

    good_new = p1[st == 1]
    good_old = p0[st == 1]

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(fr1, cmap='gray')
    for new, old in zip(good_new, good_old):
        a, b = new.ravel()
        c, d = old.ravel()
        ax.arrow(c, d, a - c, b - d, head_width=3, head_length=3,
                  fc='red', ec='red', alpha=0.6)
    ax.set_title(f'LK 光流 ({len(good_new)}/{len(p0)} 成功追踪)')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

interact(
    explore_lk_flow,
    max_corners=IntSlider(value=50, min=10, max=200, step=10, description='最大角点数'),
    quality=FloatSlider(value=0.3, min=0.01, max=0.5, step=0.01, description='角点质量'),
    min_dist=IntSlider(value=7, min=1, max=30, step=1, description='最小间距'),
    win_size=IntSlider(value=15, min=5, max=31, step=2, description='搜索窗口'),
    max_level=IntSlider(value=2, min=0, max=5, step=1, description='金字塔层数'),
)
print('请使用上方滑块交互式探索 LK 光流参数！')


In [ ]:
# ===== MOG2 背景减除 =====

# 使用弹跳球视频
bg_subtractor = cv2.createBackgroundSubtractorMOG2(
    history=30,          # 用于建模背景的帧数
    varThreshold=25,     # 方差阈值
    detectShadows=False  # 关闭阴影检测（简化演示）
)

# 对前 30 帧应用背景减除
sample_frames_idx = [0, 10, 20, 29, 40, 55]
fg_masks = []

for i in range(len(ball_frames)):
    fg_mask = bg_subtractor.apply(ball_frames[i])
    if i in sample_frames_idx:
        fg_masks.append((i, fg_mask.copy()))

# 可视化
fig, axes = plt.subplots(2, len(fg_masks), figsize=(16, 6))

for col, (frame_idx, mask) in enumerate(fg_masks):
    # 原始帧
    axes[0, col].imshow(ball_frames[frame_idx])
    axes[0, col].set_title(f'帧 {frame_idx}', fontsize=10)
    axes[0, col].axis('off')

    # 前景掩膜
    axes[1, col].imshow(mask, cmap='gray')
    axes[1, col].set_title(f'前景掩膜 (帧{frame_idx})', fontsize=10)
    axes[1, col].axis('off')

plt.suptitle('MOG2 背景减除——各帧前景检测', fontsize=14)
plt.tight_layout()
plt.show()

print('背景减除要点：')
print('- 前几帧用于建立背景模型，前景检测可能不准确')
print('- 随着帧数增加，背景模型逐渐稳定')
print('- 物体运动时被检测为前景（白色）')
print('- 静止部分逐渐融入背景（黑色）')

## 基本目标跟踪：质心跟踪

质心跟踪（Centroid Tracking）是简单但有效的目标跟踪方法：

1. 对每一帧进行前景检测（背景减除或阈值分割）
2. 查找前景区域的轮廓
3. 计算每个轮廓的质心（中心点）
4. 将当前帧的质心与上一帧的质心按最近距离匹配
5. 记录每个目标的运动轨迹

In [ ]:
# ===== 质心跟踪实现 =====

# 生成新视频：白色矩形在暗背景上移动（更容易检测）
tracking_frames, tracking_positions = generate_moving_rect_frames(
    num_frames=40, frame_size=(200, 200), rect_size=(25, 25), velocity=(4, 3)
)

# 跟踪所有帧中的目标质心
all_centroids = []

for frame in tracking_frames:
    # 二值化
    _, binary = cv2.threshold(frame, 127, 255, cv2.THRESH_BINARY)

    # 查找轮廓
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    frame_centroids = []
    for cnt in contours:
        if cv2.contourArea(cnt) > 50:  # 过滤小噪点
            M = cv2.moments(cnt)
            if M['m00'] != 0:
                cx = int(M['m10'] / M['m00'])
                cy = int(M['m01'] / M['m00'])
                frame_centroids.append((cx, cy))

    all_centroids.append(frame_centroids)

# 提取轨迹
trajectory = [c[0] for c in all_centroids if len(c) > 0]
trajectory = np.array(trajectory)
ground_truth = np.array(tracking_positions[:len(trajectory)])

print(f'跟踪帧数: {len(trajectory)}')
print(f'起点: ({trajectory[0, 0]}, {trajectory[0, 1]})')
print(f'终点: ({trajectory[-1, 0]}, {trajectory[-1, 1]})')

# 可视化轨迹
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 显示最后一帧 + 轨迹
last_frame_color = cv2.cvtColor(tracking_frames[-1], cv2.COLOR_GRAY2RGB)
for i in range(len(trajectory) - 1):
    pt1 = tuple(trajectory[i])
    pt2 = tuple(trajectory[i + 1])
    cv2.line(last_frame_color, pt1, pt2, (0, 255, 0), 1)
cv2.circle(last_frame_color, tuple(trajectory[0]), 5, (0, 255, 0), -1)  # 起点
cv2.circle(last_frame_color, tuple(trajectory[-1]), 5, (0, 0, 255), -1) # 终点

axes[0].imshow(last_frame_color)
axes[0].set_title('最后一帧 + 运动轨迹\n(绿=起点, 红=终点)')
axes[0].axis('off')

# XY 轨迹图
axes[1].plot(trajectory[:, 0], trajectory[:, 1], 'g-', linewidth=2, label='检测轨迹')
axes[1].plot(ground_truth[:, 0], ground_truth[:, 1], 'b--', linewidth=1, alpha=0.5, label='真实轨迹')
axes[1].scatter(trajectory[0, 0], trajectory[0, 1], c='green', s=80, label='起点')
axes[1].scatter(trajectory[-1, 0], trajectory[-1, 1], c='red', s=80, label='终点')
axes[1].invert_yaxis()
axes[1].set_xlabel('X')
axes[1].set_ylabel('Y')
axes[1].set_title('质心运动轨迹')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('质心跟踪结果', fontsize=14)
plt.tight_layout()
plt.show()

# 误差分析
if len(trajectory) == len(ground_truth):
    errors = np.sqrt(((trajectory - ground_truth) ** 2).sum(axis=1))
    print(f'\n跟踪误差:')
    print(f'  平均误差: {errors.mean():.2f} 像素')
    print(f'  最大误差: {errors.max():.2f} 像素')


## 练习与实践

通过以下练习加深对视频分析和目标跟踪的理解。

In [ ]:
# 练习1: 修改合成视频的参数，观察光流算法的跟踪效果
# TODO: 完成以下练习

# 尝试不同的物体速度、大小和轨迹模式

def experiment_optical_flow(velocity_x=3, velocity_y=2, ball_radius=20, num_frames=30):
    """实验函数：修改参数观察光流效果"""
    # 生成视频
    frames, positions = generate_bouncing_ball_frames(
        num_frames=num_frames, frame_size=(300, 300),
        ball_radius=ball_radius, velocity=(velocity_x, velocity_y)
    )

    # 计算帧0和帧2之间的密集光流
    gray0 = cv2.cvtColor(frames[0], cv2.COLOR_RGB2GRAY)
    gray1 = cv2.cvtColor(frames[min(2, num_frames-1)], cv2.COLOR_RGB2GRAY)

    flow = cv2.calcOpticalFlowFarneback(gray0, gray1, None,
                                          pyr_scale=0.5, levels=3, winsize=15,
                                          iterations=3, poly_n=5, poly_sigma=1.2, flags=0)

    mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(frames[0])
    axes[0].set_title(f'帧 0\n速度=({velocity_x},{velocity_y}), 半径={ball_radius}')
    axes[0].axis('off')

    axes[1].imshow(frames[min(2, num_frames-1)])
    axes[1].set_title(f'帧 {min(2, num_frames-1)}')
    axes[1].axis('off')

    axes[2].imshow(mag, cmap='hot')
    axes[2].set_title(f'光流幅度\n最大位移={mag.max():.2f} px')
    axes[2].axis('off')

    plt.suptitle('不同参数的光流效果', fontsize=14)
    plt.tight_layout()
    plt.show()

    return mag.max()


# 测试不同参数
print('测试 1: 慢速小球')
_ = experiment_optical_flow(velocity_x=2, velocity_y=1, ball_radius=15)

print('\n测试 2: 快速大球')
_ = experiment_optical_flow(velocity_x=8, velocity_y=5, ball_radius=30)

print('\n测试 3: 快速小球')
_ = experiment_optical_flow(velocity_x=8, velocity_y=5, ball_radius=12)

print('\n观察要点：')
print('- 速度越快，光流幅度越大')
print('- 物体越大，运动区域在光流图中越明显')
print('- 纯色物体的内部光流为零（孔径问题）')

In [ ]:
# 练习2: 尝试使用CSRT或KCF跟踪器来跟踪合成视频中的物体
# TODO: 完成以下练习

# OpenCV 提供了多种目标跟踪器：
# - CSRT: 高精度但较慢 (cv2.TrackerCSRT_create)
# - KCF:  速度快，适合实时应用 (cv2.TrackerKCF_create)
# - MOSSE: 最快但精度较低

print('OpenCV 跟踪器可用性检查:')
tracker_types = []
try:
    tracker = cv2.TrackerCSRT_create()
    tracker_types.append('CSRT')
    print('  CSRT 跟踪器: 可用')
except Exception as e:
    print(f'  CSRT 跟踪器: 不可用 ({e})')

try:
    tracker = cv2.TrackerKCF_create()
    tracker_types.append('KCF')
    print('  KCF 跟踪器: 可用')
except Exception as e:
    print(f'  KCF 跟踪器: 不可用 ({e})')

# 生成测试视频
test_frames, _ = generate_moving_rect_frames(
    num_frames=40, frame_size=(300, 300), rect_size=(30, 30), velocity=(4, 3)
)

if 'CSRT' in tracker_types:
    # 初始化 CSRT 跟踪器
    tracker = cv2.TrackerCSRT_create()

    # 初始帧中的目标边界框
    init_bbox = (20, 20, 30, 30)  # (x, y, w, h)

    # 将第一帧转为彩色（跟踪器需要 3 通道）
    first_frame = cv2.cvtColor(test_frames[0], cv2.COLOR_GRAY2BGR)
    tracker.init(first_frame, init_bbox)

    print(f'\nCSRT 跟踪器已初始化，初始边界框: {init_bbox}')
    print('在多帧上测试跟踪...')

    # 在多个帧上测试
    test_indices = [10, 20, 30, 39]
    tracked_bboxes = []

    for idx in test_indices:
        frame_color = cv2.cvtColor(test_frames[idx], cv2.COLOR_GRAY2BGR)
        success, bbox = tracker.update(frame_color)
        if success:
            tracked_bboxes.append((idx, bbox))
            bbox_int = tuple(map(int, bbox))
            print(f'  帧 {idx}: 成功, 边界框={bbox_int}')
        else:
            print(f'  帧 {idx}: 跟踪失败')

    # 可视化跟踪结果
    fig, axes = plt.subplots(1, len(tracked_bboxes) + 1, figsize=(16, 4))

    # 初始帧
    axes[0].imshow(test_frames[0], cmap='gray')
    rect_init = RectPatch((init_bbox[0], init_bbox[1]), init_bbox[2], init_bbox[3],
                           fill=False, edgecolor='lime', linewidth=2)
    axes[0].add_patch(rect_init)
    axes[0].set_title('帧 0 (初始化)')
    axes[0].axis('off')

    for i, (idx, bbox) in enumerate(tracked_bboxes):
        ax = axes[i + 1]
        ax.imshow(test_frames[idx], cmap='gray')
        rect = RectPatch((bbox[0], bbox[1]), bbox[2], bbox[3],
                          fill=False, edgecolor='red', linewidth=2)
        ax.add_patch(rect)
        ax.set_title(f'帧 {idx}')
        ax.axis('off')

    plt.suptitle('CSRT 跟踪器结果', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('\n注意：当前 OpenCV 版本不支持 CSRT/KCF 跟踪器。')
    print('尝试升级: pip install opencv-contrib-python')
    print('\n替代方案：使用简单的模板匹配进行演示')

    # 简单模板匹配替代方案
    template = test_frames[0][15:45, 15:45]  # 提取目标模板
    test_indices = [10, 20, 30, 39]

    fig, axes = plt.subplots(1, len(test_indices), figsize=(16, 4))
    for i, idx in enumerate(test_indices):
        result = cv2.matchTemplate(test_frames[idx], template, cv2.TM_CCOEFF_NORMED)
        _, max_val, _, max_loc = cv2.minMaxLoc(result)
        axes[i].imshow(test_frames[idx], cmap='gray')
        rect = RectPatch((max_loc[0], max_loc[1]), template.shape[1], template.shape[0],
                          fill=False, edgecolor='red', linewidth=2)
        axes[i].add_patch(rect)
        axes[i].set_title(f'帧 {idx}\n匹配度={max_val:.3f}')
        axes[i].axis('off')

    plt.suptitle('模板匹配（KCF/CSRT 替代方案）', fontsize=14)
    plt.tight_layout()
    plt.show()

## 实战案例：交通流量监测模拟

这是一个完整的视频分析实战案例，模拟交通监控场景：

1. **生成合成交通视频**：多条车道，多个车辆在不同方向移动
2. **背景减除**：使用 MOG2 检测运动车辆
3. **车辆检测**：查找前景区域轮廓，过滤噪声
4. **车辆计数**：通过虚拟检测线统计通过车辆数
5. **可视化**：在视频帧上绘制检测框和计数信息

In [ ]:
# ===== 生成合成交通视频 =====

def generate_traffic_video(num_frames=60, frame_size=(500, 300)):
    """生成多车道交通模拟视频"""
    h, w = frame_size

    # 绘制道路
    road = np.zeros((h, w, 3), dtype=np.uint8)
    # 灰色路面
    road[:, :] = [80, 80, 80]
    # 车道线
    for lane_y in [h//4, h//2, 3*h//4]:
        for x in range(0, w, 30):
            cv2.rectangle(road, (x, lane_y - 1), (x + 15, lane_y + 1), (200, 200, 200), -1)
    # 边线
    cv2.line(road, (0, h//8), (w, h//8), (255, 255, 255), 3)
    cv2.line(road, (0, 7*h//8), (w, 7*h//8), (255, 255, 255), 3)

    # 车辆参数：(初始x, y车道, 速度, 颜色, 大小)
    vehicles = [
        {'x': -30, 'lane': 0, 'speed': 3, 'color': (220, 50, 50), 'size': (35, 20), 'counted': False},
        {'x': w + 30, 'lane': 1, 'speed': -2, 'color': (50, 50, 220), 'size': (30, 18), 'counted': False},
        {'x': -40, 'lane': 2, 'speed': 4, 'color': (50, 200, 50), 'size': (40, 22), 'counted': False},
        {'x': w + 50, 'lane': 0, 'speed': -3, 'color': (220, 200, 50), 'size': (32, 18), 'counted': False},
    ]

    # 新增车辆的时机
    spawn_schedule = {20: {'lane': 2, 'speed': 5, 'color': (200, 100, 50), 'size': (36, 20)},
                       35: {'lane': 1, 'speed': -4, 'color': (100, 200, 200), 'size': (28, 18)}}

    frames = []
    vehicle_counts = []
    total_count = 0

    # 检测线位置
    detection_line_x = w // 3

    for frame_idx in range(num_frames):
        frame = road.copy()

        # 画检测线
        cv2.line(frame, (detection_line_x, 0), (detection_line_x, h), (0, 255, 255), 2)

        # 定时生成新车辆
        if frame_idx in spawn_schedule:
            new_v = spawn_schedule[frame_idx]
            new_from = -40 if new_v['speed'] > 0 else w + 40
            vehicles.append({
                'x': new_from, 'lane': new_v['lane'],
                'speed': new_v['speed'], 'color': new_v['color'],
                'size': new_v['size'], 'counted': False,
            })

        # 更新和绘制车辆
        lane_y_positions = [h//4 - 20, h//2 - 18, 3*h//4 - 22]

        for v in vehicles:
            v['x'] += v['speed']
            lane_y = lane_y_positions[v['lane']]
            car_w, car_h = v['size']

            # 绘制车辆
            x1 = int(v['x'])
            y1 = lane_y
            x2 = max(0, min(x1 + car_w, w - 1))
            cv2.rectangle(frame, (max(0, x1), y1), (x2, y1 + car_h), v['color'], -1)
            cv2.rectangle(frame, (max(0, x1), y1), (x2, y1 + car_h), (255, 255, 255), 1)

            # 检测线计数
            prev_x = v['x'] - v['speed']
            if not v['counted'] and prev_x < detection_line_x <= v['x']:
                v['counted'] = True
                total_count += 1

        # 移除驶出画面的车辆
        vehicles = [v for v in vehicles if -60 < v['x'] < w + 60]

        # 在帧上添加计数信息
        cv2.putText(frame, f'Vehicle Count: {total_count}', (10, h - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(frame, f'Frame: {frame_idx}', (w - 120, h - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        frames.append(frame)
        vehicle_counts.append(total_count)

    return frames, vehicle_counts


# 生成交通视频
traffic_frames, traffic_counts = generate_traffic_video(num_frames=60, frame_size=(500, 300))
print(f'生成帧数: {len(traffic_frames)}')
print(f'最终车辆计数: {traffic_counts[-1]}')

# 显示一些帧
sample_idx = [0, 15, 30, 45, 59]
fig, axes = plt.subplots(1, len(sample_idx), figsize=(18, 4))
for ax, idx in zip(axes, sample_idx):
    ax.imshow(traffic_frames[idx])
    ax.set_title(f'帧 {idx}\n计数={traffic_counts[idx]}')
    ax.axis('off')
plt.suptitle('合成交通监控视频', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ===== 交通流量检测完整流程 =====

# 对交通视频应用 MOG2 背景减除
bg_sub = cv2.createBackgroundSubtractorMOG2(history=30, varThreshold=30, detectShadows=False)

detected_frames = []
all_detections = []

for i, frame in enumerate(traffic_frames):
    # 背景减除
    fg_mask = bg_sub.apply(frame)

    # 形态学清理
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    fg_clean = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN, kernel, iterations=2)
    fg_clean = cv2.morphologyEx(fg_clean, cv2.MORPH_CLOSE, kernel, iterations=2)

    # 查找前景轮廓
    contours, _ = cv2.findContours(fg_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # 过滤小轮廓（噪声）
    min_area = 200
    vehicles_detected = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > min_area:
            x, y, w, h = cv2.boundingRect(cnt)
            vehicles_detected.append((x, y, x + w, y + h, area))

    # 在帧上绘制检测结果
    vis_frame = frame.copy()
    for (x1, y1, x2, y2, area) in vehicles_detected:
        cv2.rectangle(vis_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(vis_frame, f'{area:.0f}px', (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

    detected_frames.append(vis_frame)
    all_detections.append(vehicles_detected)

# 展示结果
sample_idx = [15, 30, 45, 59]
fig, axes = plt.subplots(2, len(sample_idx), figsize=(16, 8))

for col, idx in enumerate(sample_idx):
    axes[0, col].imshow(traffic_frames[idx])
    axes[0, col].set_title(f'帧 {idx} (原始)')
    axes[0, col].axis('off')

    axes[1, col].imshow(detected_frames[idx])
    nd = len(all_detections[idx])
    axes[1, col].set_title(f'帧 {idx} (检测到 {nd} 辆车)')
    axes[1, col].axis('off')

plt.suptitle('交通流量监测——车辆检测结果', fontsize=14)
plt.tight_layout()
plt.show()

# 绘制车辆计数随时间变化
plt.figure(figsize=(10, 5))
plt.plot(range(len(traffic_counts)), traffic_counts, 'b-o', markersize=3, linewidth=2)
plt.xlabel('帧编号')
plt.ylabel('累计车辆数')
plt.title('车辆计数随时间变化')
plt.grid(True, alpha=0.3)
plt.show()

print(f'\n===== 交通监测汇总 =====')
print(f'总帧数: {len(traffic_frames)}')
print(f'累计通过车辆数: {traffic_counts[-1]}')
print(f'平均每帧检测车辆数: {np.mean([len(d) for d in all_detections]):.1f}')
print(f'最大同时出现车辆数: {max([len(d) for d in all_detections])}')


In [ ]:
# ===== 交通流量最终可视化展示 =====

# 创建一张综合展示图
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 展示多个时刻的检测结果
show_frames = [0, 20, 40, 59]
for i, frame_idx in enumerate(show_frames):
    axes[0, i].imshow(traffic_frames[frame_idx])
    axes[0, i].set_title(f'原始帧 {frame_idx}')
    axes[0, i].axis('off')

# 对应的检测结果
for i, frame_idx in enumerate(show_frames):
    axes[1, i].imshow(detected_frames[frame_idx])
    vehicles_at_frame = len(all_detections[frame_idx])
    axes[1, i].set_title(f'检测帧 {frame_idx} ({vehicles_at_frame}辆车)')
    axes[1, i].axis('off')

# 累计计数图
axes[0, 2].plot(range(len(traffic_counts)), traffic_counts, 'b-', linewidth=2)
axes[0, 2].set_xlabel('帧编号')
axes[0, 2].set_ylabel('累计车辆数')
axes[0, 2].set_title('累计车辆计数')
axes[0, 2].grid(True, alpha=0.3)

# 每帧检测数
per_frame_counts = [len(d) for d in all_detections]
axes[1, 2].bar(range(len(per_frame_counts)), per_frame_counts, color='steelblue', alpha=0.7)
axes[1, 2].set_xlabel('帧编号')
axes[1, 2].set_ylabel('检测车辆数')
axes[1, 2].set_title('每帧检测到的车辆数')
axes[1, 2].set_ylim(0, max(per_frame_counts) + 1)
axes[1, 2].grid(True, alpha=0.3, axis='y')

plt.suptitle('交通流量监测——完整模拟', fontsize=16)
plt.tight_layout()
plt.show()

print('\n交通流量监测模拟总结：')
print('1. 使用合成视频生成了多车道交通场景')
print('2. MOG2 背景减除有效检测到了运动车辆')
print('3. 通过虚拟检测线实现了车辆计数')
print('4. 形态学操作清理了检测噪声')
print('5. 此方法可扩展到真实交通摄像头视频')

## 总结与延伸

### 本模块核心要点

| 知识点 | 核心概念 | 关键函数/类 |
|--------|---------|-----------|
| 稀疏光流 | LK 金字塔，只在角点计算 | calcOpticalFlowPyrLK, goodFeaturesToTrack |
| 稠密光流 | Farneback 方法，逐像素计算 | calcOpticalFlowFarneback |
| 目标跟踪 | 单目标/多目标框跟踪 | TrackerCSRT_create, TrackerKCF_create |
| 背景减除 | MOG2 (混合高斯) / KNN | createBackgroundSubtractorMOG2 |
| 质心跟踪 | 最近距离匹配 | findContours + moments |

### 关键技巧

- 光流依赖角点质量：好的角点检测是好光流的前提
- 对象跟踪使用 KCF 平衡速度和精度，CSRT 精度最高但较慢
- 背景减除需要先建立背景模型（让算法多看几帧）
- 前景 mask 通常需要形态学后处理去除噪点

### 延伸阅读

- 《Learning OpenCV 3》-- Adrian Kaehler & Gary Bradski 第 16-17 章
- OpenCV 官方 Tracking API 教程
- Deep SORT（基于深度学习的多目标跟踪）
- YOLO + 跟踪（检测 + 跟踪的现代范式）
- 3D 光流与场景流

### 课程回顾

至此，我们完成了从几何变换（模块03）、图像分割与形态学（模块04）到视频分析与目标跟踪（模块05）的学习。下一个模块将进入深度学习的世界：CNN 基础与图像分类。